In [ ]:
# ==============================================================================
# PARALLEL DIM: CUSTOMER + EQUIPMENT
# ==============================================================================
from notebooks.helpers import (
    IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger,
    safe_count, generate_batch_id,
)
from notebooks.helpers.silver_transforms import (
    transform_customer_full_pipeline,
    transform_equipment_dimension,
)
import pandas as pd

logger = setup_logger("parallel_dim_customer_equipment")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

customer_batch_id = get_latest_batch_id(spark, "customer")
if not customer_batch_id:
    raise ValueError("No bronze batch_id found for customer; run bronze load first.")
logger.info(f"Using bronze batch_id for customer: {customer_batch_id}")

equipment_batch_id = get_latest_batch_id(spark, "equipment")
if not equipment_batch_id:
    raise ValueError("No bronze batch_id found for equipment; run bronze load first.")
logger.info(f"Using bronze batch_id for equipment: {equipment_batch_id}")

customer_config = TableConfig(
    table_name="customer",
    business_key="customer_id",
    surrogate_key="customer_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_customer",
    silver_transform=transform_customer_full_pipeline,
    dependencies=["address", "city", "country"],
)

equipment_config = TableConfig(
    table_name="equipment",
    business_key="equipment_id",
    surrogate_key="equipment_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_equipment",
    silver_transform=transform_equipment_dimension,
)

print("Row Counts (Before):")
print(f"dim_customer: {safe_count(spark, 'dim_customer')}")
print(f"dim_equipment: {safe_count(spark, 'dim_equipment')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))


In [ ]:
results = []
results += pipeline.load_tables([customer_config], force_full=False, bronze_batch_id=customer_batch_id)
results += pipeline.load_tables([equipment_config], force_full=False, bronze_batch_id=equipment_batch_id)

display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_customer: {safe_count(spark, 'dim_customer')}")
print(f"dim_equipment: {safe_count(spark, 'dim_equipment')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
